In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
train_df = pd.read_csv("data/train.csv",encoding="utf-8")
test_df = pd.read_csv("data/test.csv",encoding="utf-8")

In [3]:
import os
print(os.getcwd())

C:\Users\Berkay\PycharmProjects\sandbox\titanic_comp


In [4]:
train_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
train_df.count()

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

In [6]:
train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
title_map = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mrs": "Mrs",
    "Mme": "Mrs",
    "Master": "Master",
    "Rev": "Rare",
    "Dr": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Don": "Rare",
    "Dona": "Rare",
    "Lady": "Rare",
    "Sir": "Rare",
    "Capt": "Rare",
    "the Countess": "Rare",
    "Jonkheer": "Rare",
}

In [8]:
def preprocess_df(df):

    df = df.copy()

    drop_cols = ["PassengerId","Name","Ticket",]

    #get title
    get_name = df["Name"].apply(lambda x: x.split(",")[1])
    get_title = get_name.apply(lambda x: x.split(".")[0])
    df["Title"] = get_title
    df["Title"] = df["Title"].str.strip().map(title_map)

    #get Cabin title
    cabin_list = df["Cabin"].astype(str).to_list()
    cabins = []
    for x in cabin_list:
        if str(x) == "nan":
            cabins.append("Unknown")
        else:
            cabins.append(x[0])
    df["Cabin_Title"] = cabins

    #Embarked
    df["Embarked"] = df["Embarked"].fillna("S") ##fill embarked na with S as its the most common one

    #Gender
    df["Sex"] = df["Sex"].map({"male":1, "female":0}) ## map sex col to 0 and 1

    #Ticket
    df['TicketCount'] = df.groupby('Ticket')['Ticket'].transform('count') ##counts how many people holding the same ticket
    df = df.drop(columns='Ticket') ##drops Ticket after

    #add features
    df["Family_size"] = df["SibSp"] + df["Parch"] + 1
    df["isAlone"] = (df["Family_size"] == 1).astype(int)
    df["GirlWithParents"] = ((df["Parch"] > 0) & (df["Title"] == "Miss")).astype(int)

    df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True, dtype=int) ##turn the columns into one hot

    return df


In [9]:
train_pro_df = preprocess_df(train_df)

In [10]:
train_df.groupby(["Sex","Embarked"]).count()

PassengerId  Survived  Pclass  Name  Age  SibSp  Parch  \
Sex    Embarked                                                           
female C                  73        73      73    73   61     73     73   
       Q                  36        36      36    36   12     36     36   
       S                 203       203     203   203  186    203    203   
male   C                  95        95      95    95   69     95     95   
       Q                  41        41      41    41   16     41     41   
       S                 441       441     441   441  368    441    441   

                 Ticket  Fare  Cabin  
Sex    Embarked                       
female C             73    73     37  
       Q             36    36      2  
       S            203   203     56  
male   C             95    95     32  
       Q             41    41      2  
       S            441   441     73

In [11]:
train_df.iloc[train_df["Embarked"].isna()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [12]:
train_pro_df.isna().sum()

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                177
SibSp                0
Parch                0
Fare                 0
Cabin              687
Cabin_Title          0
TicketCount          0
Family_size          0
isAlone              0
GirlWithParents      0
Embarked_Q           0
Embarked_S           0
Title_Miss           0
Title_Mr             0
Title_Mrs            0
Title_Rare           0
dtype: int64

In [13]:
res_cabin = train_pro_df.drop(["Cabin","Name","PassengerId","Cabin_Title"], axis=1)

In [14]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Family_size,isAlone,GirlWithParents,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,7.2500,1,2,0,0,0,1,0,1,0,0
1,1,1,0,38.0,1,0,71.2833,1,2,0,0,0,0,0,0,1,0
2,1,3,0,26.0,0,0,7.9250,1,1,1,0,0,1,1,0,0,0
3,1,1,0,35.0,1,0,53.1000,2,2,0,0,0,1,0,0,1,0
4,0,3,1,35.0,0,0,8.0500,1,1,1,0,0,1,0,1,0,0


In [15]:
from sklearn.preprocessing import StandardScaler

fare_scaler = StandardScaler()
res_cabin["Fare"] = fare_scaler.fit_transform(res_cabin[["Fare"]])

In [16]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Family_size,isAlone,GirlWithParents,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,-0.502445,1,2,0,0,0,1,0,1,0,0
1,1,1,0,38.0,1,0,0.786845,1,2,0,0,0,0,0,0,1,0
2,1,3,0,26.0,0,0,-0.488854,1,1,1,0,0,1,1,0,0,0
3,1,1,0,35.0,1,0,0.420730,2,2,0,0,0,1,0,0,1,0
4,0,3,1,35.0,0,0,-0.486337,1,1,1,0,0,1,0,1,0,0


In [17]:
res_cabin.isna().sum()

Survived             0
Pclass               0
Sex                  0
Age                177
SibSp                0
Parch                0
Fare                 0
TicketCount          0
Family_size          0
isAlone              0
GirlWithParents      0
Embarked_Q           0
Embarked_S           0
Title_Miss           0
Title_Mr             0
Title_Mrs            0
Title_Rare           0
dtype: int64

In [18]:
age_known = res_cabin[res_cabin["Age"].notna()]
age_missing = res_cabin[res_cabin["Age"].isna()]
age_X = age_known.drop(columns=["Age", "Survived"])
age_y = age_known["Age"]

In [19]:
age_X.columns

Index(['Pclass', 'Sex', 'SibSp', 'Parch', 'Fare', 'TicketCount', 'Family_size',
       'isAlone', 'GirlWithParents', 'Embarked_Q', 'Embarked_S', 'Title_Miss',
       'Title_Mr', 'Title_Mrs', 'Title_Rare'],
      dtype='str')

In [20]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(age_X, age_y, test_size=0.2, random_state=42)

In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=500, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Model MAE:", mean_absolute_error(y_test, pred))

# baseline: just guess the median age every time
print("Baseline MAE:", mean_absolute_error(y_test, [y_train.median()] * len(y_test)))

Model MAE: 8.307943586785258
Baseline MAE: 10.804195804195805


In [22]:
print(X_train.nunique()[X_train.nunique() <= 1])

Series([], dtype: int64)


In [23]:
from sklearn.ensemble import HistGradientBoostingRegressor
gb = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
print(mean_absolute_error(y_test, gb.predict(X_test)))

7.965204925337126


In [24]:
from sklearn.model_selection import cross_val_score
for name, m in [("RF", model), ("GB", gb)]:
    s = -cross_val_score(m, age_X, age_y, cv=5, scoring="neg_mean_absolute_error")
    print(name, s.mean().round(2))

RF 9.36
GB 9.0


In [25]:
gb.fit(age_X, age_y)
res_cabin.loc[res_cabin["Age"].isna(), "Age"] = gb.predict(age_missing.drop(columns=["Age", "Survived"]))

In [26]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Family_size,isAlone,GirlWithParents,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,-0.502445,1,2,0,0,0,1,0,1,0,0
1,1,1,0,38.0,1,0,0.786845,1,2,0,0,0,0,0,0,1,0
2,1,3,0,26.0,0,0,-0.488854,1,1,1,0,0,1,1,0,0,0
3,1,1,0,35.0,1,0,0.420730,2,2,0,0,0,1,0,0,1,0
4,0,3,1,35.0,0,0,-0.486337,1,1,1,0,0,1,0,1,0,0


In [27]:
res_cabin.isna().sum()

Survived           0
Pclass             0
Sex                0
Age                0
SibSp              0
Parch              0
Fare               0
TicketCount        0
Family_size        0
isAlone            0
GirlWithParents    0
Embarked_Q         0
Embarked_S         0
Title_Miss         0
Title_Mr           0
Title_Mrs          0
Title_Rare         0
dtype: int64

In [28]:
res_cabin.shape

(891, 17)

In [29]:
res_cabin["Cabin_Title"] = train_pro_df["Cabin_Title"]

In [30]:
res_cabin["Cabin_Title"] = res_cabin["Cabin_Title"].replace("Unknown", np.nan)

In [31]:
cabin_df = res_cabin.copy()

In [32]:
cabin_missing = cabin_df[cabin_df["Cabin_Title"].isna()]
cabin_known = cabin_df[cabin_df["Cabin_Title"].notna()]

cabin_X = cabin_known.drop(["Cabin_Title"],axis=1)
cabin_y = cabin_known["Cabin_Title"]

In [33]:
X_train,X_test,y_train,y_test = train_test_split(cabin_X, cabin_y, test_size=0.2, random_state=42)

In [35]:
from sklearn.ensemble import HistGradientBoostingClassifier

res_cabin["HasCabin"] = res_cabin["Cabin_Title"].notna().astype(int)
surv = res_cabin.drop("Cabin_Title", axis=1)

X = surv.drop(columns="Survived")
y = surv["Survived"]
clf = HistGradientBoostingClassifier(random_state=42)

print("With HasCabin:   ", cross_val_score(clf, X, y, cv=5).mean().round(3))

With HasCabin:    0.829


In [39]:
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import loguniform, randint

params = {
    "learning_rate": loguniform(0.01, 0.3),
    "max_depth": [None, 3, 4, 5, 7],
    "min_samples_leaf": randint(15, 100),
    "l2_regularization": loguniform(0.1, 20),
    "max_iter": randint(100, 500),
    "max_leaf_nodes": [15, 31, 63],
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42), params,
    n_iter=300, cv=cv, scoring="accuracy", n_jobs=-1, random_state=42, verbose=1,
)
search.fit(X, y)
print(search.best_params_, search.best_score_.round(3))

Fitting 15 folds for each of 300 candidates, totalling 4500 fits
{'l2_regularization': np.float64(6.362418276592956), 'learning_rate': np.float64(0.05132787337939953), 'max_depth': None, 'max_iter': 287, 'max_leaf_nodes': 31, 'min_samples_leaf': 28} 0.838


In [40]:
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform, randint

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

models = {
    "hgb": (HistGradientBoostingClassifier(random_state=42), {
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": [None, 3, 4, 5, 7],
        "min_samples_leaf": randint(15, 100),
        "l2_regularization": loguniform(0.1, 20),
        "max_iter": randint(100, 500),
        "max_leaf_nodes": [15, 31, 63],
    }, 150),
    "rf": (RandomForestClassifier(random_state=42, n_jobs=-1), {
        "n_estimators": randint(200, 800),
        "max_depth": [None, 4, 6, 8, 10],
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", 0.5, None],
    }, 60),
    "lr": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), {
        "logisticregression__C": loguniform(0.01, 100),
    }, 40),
    "svc": (make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)), {
        "svc__C": loguniform(0.1, 100),
        "svc__gamma": loguniform(1e-3, 1),
    }, 80),
}

best = {}
for name, (est, params, n_iter) in models.items():
    s = RandomizedSearchCV(est, params, n_iter=n_iter, cv=cv, scoring="accuracy",
                           n_jobs=-1, random_state=42, verbose=1)
    s.fit(X, y)
    best[name] = s.best_estimator_
    print(f"{name}: {s.best_score_:.3f}  {s.best_params_}")

for voting in ["hard", "soft"]:
    vote = VotingClassifier(list(best.items()), voting=voting, n_jobs=-1)
    print(f"vote_{voting}: {cross_val_score(vote, X, y, cv=cv).mean():.3f}")

Fitting 15 folds for each of 150 candidates, totalling 2250 fits
hgb: 0.834  {'l2_regularization': np.float64(9.891481041192067), 'learning_rate': np.float64(0.06759977233086435), 'max_depth': 5, 'max_iter': 468, 'max_leaf_nodes': 63, 'min_samples_leaf': 27}
Fitting 15 folds for each of 60 candidates, totalling 900 fits
rf: 0.832  {'max_depth': None, 'max_features': None, 'min_samples_leaf': 5, 'n_estimators': 661}
Fitting 15 folds for each of 40 candidates, totalling 600 fits
lr: 0.829  {'logisticregression__C': np.float64(13.826232179369875)}
Fitting 15 folds for each of 80 candidates, totalling 1200 fits


C:\Users\Berkay\PycharmProjects\sandbox\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svc: 0.832  {'svc__C': np.float64(3.699972431463808), 'svc__gamma': np.float64(0.019170041589170652)}
vote_hard: 0.839
vote_soft: 0.840


In [37]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    HistGradientBoostingClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=True, n_jobs=-1,
)
bag_clf.fit(X, y)
print(cross_val_score(bag_clf, X, y, cv=5, scoring="accuracy").mean().round(3))

0.825


In [41]:
vote = VotingClassifier(list(best.items()), voting="soft", n_jobs=-1).fit(X, y)

test_pro = preprocess_df(test_df)
test_pro["Fare"] = test_pro["Fare"].fillna(train_df["Fare"].median())
test_pro["Fare"] = fare_scaler.transform(test_pro[["Fare"]])

missing = test_pro["Age"].isna()
test_pro.loc[missing, "Age"] = gb.predict(test_pro.loc[missing].reindex(columns=age_X.columns, fill_value=0))

X_sub = test_pro.reindex(columns=X.columns, fill_value=0)
print("NaNs left:", X_sub.isna().sum().sum())  # should be 0

sub = pd.DataFrame({"PassengerId": test_df["PassengerId"], "Survived": vote.predict(X_sub)})
sub.to_csv("submission.csv", index=False)

NaNs left: 0
